# UlasRasa - Training ABSA IndoBERT (Colab)

Notebook ini mereplikasi **persis** pipeline training lokal UlasRasa (`training/prepare_pairs.py`, `training/train_absa.py`, `training/predict_holdout.py` + `scripts/evaluate.py`):

1. Dataset: `sample_reviews.csv` (100 ulasan berlabel 5 aspek) + `holdout_ids.json` (25 ulasan FROZEN, seed 42).
2. Formulasi: input pasangan (teks ulasan, nama aspek sebagai `text_pair`) -> 4 kelas (`0=not_mentioned, 1=positive, 2=neutral, 3=negative`).
3. Training hanya dari 75 ulasan NON-holdout; pasangan `not_mentioned` di-subsample maks 1,2x pasangan mentioned (seed 42) + class weight.
4. Latih **Lite** (`indobenchmark/indobert-lite-base-p1`) dulu, lalu **Base** (`indobenchmark/indobert-base-p1`); max_len 128, maks 5 epoch, early stopping pada Macro-F1 validasi internal (15% train).
5. Evaluasi pada hold-out (25 x 5 pasangan, tanpa subsample): Macro-F1 per aspek + confusion matrix.

Aktifkan GPU dulu: Runtime > Change runtime type > GPU.

In [ ]:
# 1. Install dependencies
!pip -q install transformers scikit-learn pandas numpy accelerate
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

## 2. Muat dataset

Upload `sample_reviews.csv` dan `holdout_ids.json` dari repo (folder `data/`) lewat cell di bawah, atau clone repo lalu sesuaikan path.

In [ ]:
# 2. Upload dataset (atau ganti dengan path clone repo)
from google.colab import files
import pandas as pd, json

up = files.upload()  # pilih sample_reviews.csv dan holdout_ids.json
df = pd.read_csv('sample_reviews.csv')
df['review_id'] = df['review_id'].astype(str)

raw = json.load(open('holdout_ids.json'))
ids = []
if isinstance(raw, dict) and isinstance(raw.get('per_restaurant'), dict):
    for e in raw['per_restaurant'].values():
        ids += [str(x) for x in e['review_ids']]
else:
    ids = [str(x) for x in raw]
holdout_ids = ids
assert len(holdout_ids) == 25
print('total', len(df), 'holdout', len(holdout_ids))

In [ ]:
# 3. Prepare pairs (sama dengan prepare_pairs.py)
import numpy as np
ASPECTS = ['rasa','porsi','harga','kemasan','pelayanan']
train_df = df[~df['review_id'].isin(holdout_ids)]
assert len(train_df) == 75
rows = []
for _, r in train_df.iterrows():
    for asp in ASPECTS:
        rows.append({'review_id': r['review_id'], 'aspect': asp,
                     'text': str(r['review_text']), 'label': str(r[asp]).strip()})
pairs = pd.DataFrame(rows)
mentioned = int((pairs['label'] != 'not_mentioned').sum())
cap = int(1.2 * mentioned)
nm = pairs.index[pairs['label'] == 'not_mentioned'].to_numpy()
if len(nm) > cap:
    keep = np.random.RandomState(42).choice(nm, size=cap, replace=False)
    pairs = pairs[(pairs['label'] != 'not_mentioned') | pairs.index.isin(keep)]
pairs = pairs.sample(frac=1.0, random_state=42).reset_index(drop=True)
assert not set(pairs['review_id']) & set(holdout_ids), 'KEBOCORAN HOLD-OUT'
print(pairs['label'].value_counts().to_dict())
pairs.to_csv('train_pairs.csv', index=False)

## 4. Training

Fungsi `train_one` identik dengan `train_absa.py`: weighted loss, early stopping (patience 2), lr Lite 3e-5 / Base 2e-5, batch Lite 16 / Base 8.

In [ ]:
# 4. Fungsi training (identik train_absa.py)
import numpy as np, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

LABEL2ID = {'not_mentioned':0,'positive':1,'neutral':2,'negative':3}
ID2LABEL = {v:k for k,v in LABEL2ID.items()}

class DS(torch.utils.data.Dataset):
    def __init__(s, enc, lab): s.enc, s.lab = enc, lab
    def __len__(s): return len(s.lab)
    def __getitem__(s, i): return {**{k: torch.tensor(v[i]) for k,v in s.enc.items()}, 'labels': torch.tensor(s.lab[i])}

class WT(Trainer):
    def __init__(s, *a, cw=None, **kw): super().__init__(*a, **kw); s.cw = cw
    def compute_loss(s, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels'); out = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=s.cw.to(out.logits.device))(out.logits.view(-1,4), labels.view(-1))
        return (loss, out) if return_outputs else loss

def metrics(ep):
    return {'macro_f1': f1_score(ep.label_ids, np.argmax(ep.predictions,-1), average='macro', zero_division=0)}

def train_one(base_model, out_dir, lr, batch):
    data = pd.read_csv('train_pairs.csv'); data['label_id'] = data['label'].map(LABEL2ID)
    tr, va = train_test_split(data, test_size=0.15, random_state=42, stratify=data['label_id'])
    tok = AutoTokenizer.from_pretrained(base_model)
    enc = lambda f: tok(f['text'].astype(str).tolist(), f['aspect'].astype(str).tolist(),
                        truncation=True, max_length=128, padding='max_length')
    counts = data['label_id'].value_counts().sort_index()
    w = torch.tensor([len(data)/(4*counts.get(i,1)) for i in range(4)], dtype=torch.float)
    model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(output_dir=out_dir+'_ckpt', per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch, learning_rate=lr, num_train_epochs=5,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='macro_f1', seed=42, fp16=torch.cuda.is_available(),
        logging_steps=10, report_to=[], save_total_limit=2)
    t = WT(model=model, args=args, train_dataset=DS(enc(tr), tr['label_id'].tolist()),
           eval_dataset=DS(enc(va), va['label_id'].tolist()), compute_metrics=metrics,
           cw=w, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    t.train(); print(out_dir, t.evaluate()); t.save_model(out_dir); tok.save_pretrained(out_dir)

In [ ]:
# 5a. Latih Lite dulu
train_one('indobenchmark/indobert-lite-base-p1', 'indobert-lite-absa', lr=3e-5, batch=16)

In [ ]:
# 5b. Lalu Base
train_one('indobenchmark/indobert-base-p1', 'indobert-base-absa', lr=2e-5, batch=8)

## 6. Evaluasi hold-out

Prediksi 25 x 5 pasangan hold-out (tanpa subsample), tulis file prediksi format `scripts/evaluate.py`, lalu hitung metrik per aspek (urutan kelas confusion matrix: positive, neutral, negative, not_mentioned).

In [ ]:
# 6. Prediksi + evaluasi hold-out
import numpy as np, torch
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
CM = ['positive','neutral','negative','not_mentioned']
hold = df[df['review_id'].isin(holdout_ids)]

def predict_all(model_dir):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).cuda().eval()
    pm = {}
    for _, r in hold.iterrows():
        for asp in ASPECTS:
            enc = tok(str(r['review_text']), asp, truncation=True, max_length=128, return_tensors='pt').to('cuda')
            with torch.no_grad():
                pm[(r['review_id'], asp)] = ID2LABEL[int(torch.argmax(mdl(**enc).logits, -1))]
    return pm

def evaluate(pm):
    res = {}
    for asp in ASPECTS:
        yt = hold[asp].astype(str).tolist(); yp = [pm[(rid, asp)] for rid in hold['review_id']]
        p, r, f, _ = precision_recall_fscore_support(yt, yp, labels=CM, zero_division=0)
        res[asp] = {'macro_f1': round(float(np.mean(f)), 4),
                    'confusion_matrix': confusion_matrix(yt, yp, labels=CM).tolist()}
    res['macro_f1_mean'] = round(float(np.mean([res[a]['macro_f1'] for a in ASPECTS])), 4)
    return res

results = {}
for name, d in [('indobert_lite','indobert-lite-absa'), ('indobert_base','indobert-base-absa')]:
    pm = predict_all(d)
    rows = [{'review_id': rid, **{a: pm[(rid, a)] for a in ASPECTS}} for rid in hold['review_id']]
    pd.DataFrame(rows).to_csv(f'predictions_{name}.csv', index=False)
    results[name] = evaluate(pm)
    print(name, results[name]['macro_f1_mean'], {a: results[name][a]['macro_f1'] for a in ASPECTS})
json.dump(results, open('evaluation_results_colab.json','w'), indent=2)

In [ ]:
# 7. Unduh artefak (model + prediksi + metrik)
import shutil
for d in ['indobert-lite-absa','indobert-base-absa']:
    shutil.make_archive(d, 'zip', d)
    files.download(d + '.zip')
files.download('evaluation_results_colab.json')